# Learning-rate schedules and regularisation

**JobReady AI lab.** Run each cell with Shift+Enter. Try the exercises before opening the solutions at the end.

**You will:**
- Overfit a single batch to test your code
- Compare a constant learning rate with warm-up + cosine decay
- See dropout and weight decay reduce overfitting

In [ ]:
import torch, torch.nn as nn, math
torch.manual_seed(0)
X = torch.randn(600, 20); w_true = torch.randn(20, 1)
y = (X @ w_true + 0.5 * torch.randn(600, 1) > 0).float().squeeze()
Xtr, ytr, Xva, yva = X[:100], y[:100], X[100:], y[100:]   # small training set → easy to overfit

## 1. Overfit one batch (a bug check)

In [ ]:
m = nn.Sequential(nn.Linear(20, 128), nn.ReLU(), nn.Linear(128, 2))
opt = torch.optim.Adam(m.parameters(), 1e-2)
xb, yb = Xtr[:32], ytr[:32].long()
for i in range(300):
    loss = nn.functional.cross_entropy(m(xb), yb); opt.zero_grad(); loss.backward(); opt.step()
print('loss on one batch:', loss.item())   # should be close to 0

## 2. Regularisation and a warm-up + cosine schedule

In [ ]:
def fit(dropout=0.0, wd=0.0, schedule=False, epochs=200):
    m = nn.Sequential(nn.Linear(20, 256), nn.ReLU(), nn.Dropout(dropout), nn.Linear(256, 2))
    opt = torch.optim.AdamW(m.parameters(), lr=3e-3, weight_decay=wd)
    for ep in range(epochs):
        if schedule:
            warm = 10
            lr = 3e-3 * (ep + 1) / warm if ep < warm else 3e-3 * 0.5 * (1 + math.cos(math.pi * (ep - warm) / (epochs - warm)))
            for g in opt.param_groups: g['lr'] = lr
        m.train(); loss = nn.functional.cross_entropy(m(Xtr), ytr.long()); opt.zero_grad(); loss.backward(); opt.step()
    m.eval()
    with torch.no_grad():
        return (m(Xtr).argmax(1) == ytr).float().mean().item(), (m(Xva).argmax(1) == yva).float().mean().item()
for cfg in [dict(), dict(dropout=0.5), dict(wd=0.1), dict(dropout=0.5, wd=0.1, schedule=True)]:
    tr, va = fit(**cfg); print(cfg, f'train {tr:.2f}  val {va:.2f}')

## Exercises
1. Pass softmax probabilities into `cross_entropy` instead of logits. What happens to training?
2. Try SGD with momentum 0.9 instead of AdamW.

In [ ]:
# your work here

## Solutions
1. `cross_entropy` applies log-softmax itself, so applying softmax first squashes the gradients and training slows or stalls.
2. `torch.optim.SGD(m.parameters(), lr=0.05, momentum=0.9, weight_decay=wd)` usually needs a higher learning rate than AdamW.